# 🧠 Module 15 — Recursion, Algorithms & Complexity

## 🎯 What you will learn
1. Recursion: base case, recursive case, the call stack
2. Classic recursive problems (factorial, Fibonacci, power, Hanoi, permutations, subsets, backtracking)
3. Memoisation & dynamic programming
4. Big-O notation & measuring performance
5. Searching: linear, binary
6. Sorting: bubble, selection, insertion, merge, quick (with complexities)
7. Two pointers, sliding window, prefix sums, and other patterns
8. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐⭐ (recursion programs + sorting/searching are staples)

---
## 1. Recursion
A function that **calls itself** to solve a smaller version of the same problem.

Every correct recursive function has:
1. **Base case** – stops the recursion
2. **Recursive case** – reduces the problem toward the base case

In [1]:
def countdown(n):
    if n == 0:                 # base case
        print("Liftoff! 🚀")
        return
    print(n)
    countdown(n - 1)           # smaller problem

countdown(3)

3
2
1
Liftoff! 🚀


### The call stack — what really happens
`factorial(4)` builds a stack of waiting calls, then unwinds:

```
factorial(4)
 └─ 4 * factorial(3)
        └─ 3 * factorial(2)
               └─ 2 * factorial(1)
                      └─ returns 1     ← base case
               ← returns 2 * 1 = 2
        ← returns 3 * 2 = 6
 ← returns 4 * 6 = 24
```

In [2]:
def factorial(n, depth=0):
    print("  " * depth + f"factorial({n})")
    if n <= 1:
        return 1
    result = n * factorial(n - 1, depth + 1)
    print("  " * depth + f"-> {result}")
    return result

print(factorial(4))

factorial(4)
  factorial(3)
    factorial(2)
      factorial(1)
    -> 2
  -> 6
-> 24
24


### Missing base case → `RecursionError`

In [3]:
def forever(n):
    return forever(n + 1)
try:
    forever(0)
except RecursionError as e:
    print("RecursionError:", str(e)[:50])

import sys
print("limit:", sys.getrecursionlimit())

RecursionError: maximum recursion depth exceeded
limit: 1000


## 2. Classic recursion problems

In [4]:
# Sum of digits
def digit_sum(n):
    return n if n < 10 else n % 10 + digit_sum(n // 10)

# Power (fast exponentiation, O(log n))
def power(b, e):
    if e == 0: return 1
    half = power(b, e // 2)
    return half * half if e % 2 == 0 else half * half * b

# Reverse a string
def rev(s): return s if len(s) <= 1 else rev(s[1:]) + s[0]

# Palindrome
def is_pal(s): return len(s) < 2 or (s[0] == s[-1] and is_pal(s[1:-1]))

# GCD (Euclid)
def gcd(a, b): return a if b == 0 else gcd(b, a % b)

# Binary representation
def to_bin(n): return str(n) if n < 2 else to_bin(n // 2) + str(n % 2)

# Sum / max of a list
def rsum(lst): return 0 if not lst else lst[0] + rsum(lst[1:])
def rmax(lst): return lst[0] if len(lst) == 1 else max(lst[0], rmax(lst[1:]))

# Flatten arbitrarily nested lists
def flatten(x):
    return [x] if not isinstance(x, list) else [y for item in x for y in flatten(item)]

# Fibonacci (naive: exponential!)
def fib(n): return n if n < 2 else fib(n - 1) + fib(n - 2)

print(digit_sum(9875), power(2, 30), rev("python"), is_pal("racecar"), gcd(84, 36))
print(to_bin(37), rsum([1, 2, 3, 4]), rmax([3, 9, 2]), flatten([1, [2, [3, [4]]]]), fib(15))

29 1073741824 nohtyp True 12
100101 10 9 [1, 2, 3, 4] 610


### Tower of Hanoi — the recursion poster child
Move `n` disks from A to C using B. To move n: move n-1 to B, move the biggest to C, move n-1 from B to C. Needs 2ⁿ − 1 moves.

In [5]:
moves = []
def hanoi(n, src, dst, tmp):
    if n == 0:
        return
    hanoi(n - 1, src, tmp, dst)
    moves.append(f"{src}->{dst}")
    hanoi(n - 1, tmp, dst, src)

hanoi(3, "A", "C", "B")
print(len(moves), moves)

7 ['A->C', 'A->B', 'C->B', 'A->C', 'B->A', 'B->C', 'A->C']


### Generating combinatorics recursively: subsets & permutations

In [6]:
def subsets(items):
    if not items:
        return [[]]
    rest = subsets(items[1:])
    return rest + [[items[0]] + r for r in rest]

def permutations(items):
    if len(items) <= 1:
        return [items]
    out = []
    for i, x in enumerate(items):
        for p in permutations(items[:i] + items[i + 1:]):
            out.append([x] + p)
    return out

print(subsets([1, 2, 3]))
print(permutations(["a", "b", "c"]))

[[], [3], [2], [2, 3], [1], [1, 3], [1, 2], [1, 2, 3]]
[['a', 'b', 'c'], ['a', 'c', 'b'], ['b', 'a', 'c'], ['b', 'c', 'a'], ['c', 'a', 'b'], ['c', 'b', 'a']]


### Backtracking — try, and undo when stuck

In [7]:
def n_queens(n):
    solutions, cols = [], []
    def safe(r, c):
        return all(c != cc and abs(c - cc) != r - rr for rr, cc in enumerate(cols))
    def place(r):
        if r == n:
            solutions.append(cols[:]); return
        for c in range(n):
            if safe(r, c):
                cols.append(c)       # choose
                place(r + 1)         # explore
                cols.pop()           # un-choose (backtrack)
    place(0)
    return solutions
sol = n_queens(6)
print(len(n_queens(8)), "solutions for 8 queens;", "first for 6:", sol[0])

def solve_maze(grid, r=0, c=0, path=None):
    path = path or []
    if not (0 <= r < len(grid) and 0 <= c < len(grid[0])) or grid[r][c] != 0 or (r, c) in path:
        return None
    path = path + [(r, c)]
    if (r, c) == (len(grid) - 1, len(grid[0]) - 1):
        return path
    for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
        result = solve_maze(grid, r + dr, c + dc, path)
        if result: return result
maze = [[0, 1, 0], [0, 0, 0], [1, 0, 0]]
print(solve_maze(maze))

92 solutions for 8 queens; first for 6: [1, 3, 5, 0, 2, 4]
[(0, 0), (1, 0), (1, 1), (1, 2), (2, 2)]


## 3. Memoisation & Dynamic Programming
Naive `fib(35)` repeats the same sub-problems millions of times. **Cache results** (top-down memoisation) or **build from small to large** (bottom-up DP).

In [8]:
import time
from functools import lru_cache

def slow_fib(n): return n if n < 2 else slow_fib(n - 1) + slow_fib(n - 2)

@lru_cache(maxsize=None)
def fast_fib(n): return n if n < 2 else fast_fib(n - 1) + fast_fib(n - 2)

def dp_fib(n):                   # bottom-up, O(n) time, O(1) space
    a, b = 0, 1
    for _ in range(n): a, b = b, a + b
    return a

t = time.perf_counter(); slow_fib(27); slow_t = time.perf_counter() - t
t = time.perf_counter(); fast_fib(27); fast_t = time.perf_counter() - t
print(f"naive {slow_t:.3f}s vs memoised {fast_t:.6f}s")
print(dp_fib(90), fast_fib(90))

# Classic DP problems
def coin_change(coins, amount):                # fewest coins
    dp = [0] + [float("inf")] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount] if dp[amount] != float("inf") else -1

def lcs(a, b):                                 # longest common subsequence length
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[-1][-1]

def knapsack(weights, values, cap):
    dp = [0] * (cap + 1)
    for w, v in zip(weights, values):
        for c in range(cap, w - 1, -1):
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[cap]

def climb(n):                                  # ways to climb n stairs with 1 or 2 steps
    a, b = 1, 1
    for _ in range(n - 1): a, b = b, a + b
    return b

print(coin_change([1, 5, 10, 25], 63), lcs("ABCBDAB", "BDCABA"), knapsack([1, 3, 4, 5], [1, 4, 5, 7], 7), climb(10))

naive 0.047s vs memoised 0.000026s
2880067194370816120 2880067194370816120
6 4 9 89


### Recursion vs Iteration

| | Recursion | Iteration |
|---|---|---|
| Code | shorter, elegant for trees/divide-and-conquer | usually longer |
| Speed / memory | function-call overhead, stack limit | faster, O(1) extra memory |
| Termination | base case | loop condition |
| Use for | trees, backtracking, divide & conquer | simple repetition |

## 4. Big-O notation — how the cost *grows* with input size n

| Big-O | Name | Example | n = 1,000,000 |
|---|---|---|---|
| O(1) | constant | dict lookup, list index | 1 |
| O(log n) | logarithmic | binary search | 20 |
| O(n) | linear | scan a list, `sum` | 10⁶ |
| O(n log n) | linearithmic | good sorts (merge, Timsort) | 2×10⁷ |
| O(n²) | quadratic | nested loops, bubble sort | 10¹² 😱 |
| O(2ⁿ) | exponential | naive Fibonacci, subsets | ∞ |
| O(n!) | factorial | permutations, brute-force TSP | ∞∞ |

Rules: drop constants and lower-order terms (`3n² + 5n + 2` → `O(n²)`); worst case unless stated; also measure **space** complexity.

In [9]:
import timeit
def measure(stmt, setup, n=200):
    return timeit.timeit(stmt, setup, number=n)

for size in (1_000, 10_000, 100_000):
    setup = f"L=list(range({size})); S=set(L); D=dict.fromkeys(L)"
    t_list = measure(f"{size-1} in L", setup)
    t_set = measure(f"{size-1} in S", setup)
    print(f"n={size:>7}: list {t_list:.5f}s   set {t_set:.5f}s")   # list grows ~linearly; set flat

n=   1000: list 0.00177s   set 0.00002s
n=  10000: list 0.03549s   set 0.00002s
n= 100000: list 0.28933s   set 0.00002s


### Complexity of Python built-ins (average)

| Operation | list | dict / set | deque |
|---|---|---|---|
| index / lookup | O(1) | O(1) | O(n) middle |
| append / add | O(1) amortised | O(1) | O(1) |
| insert / pop at front | **O(n)** | – | **O(1)** |
| `x in c` | O(n) | O(1) | O(n) |
| sort | O(n log n) | – | – |
| slice `a[i:j]` | O(j−i) | – | – |
| `len` | O(1) | O(1) | O(1) |

## 5. Searching

In [10]:
def linear_search(a, x):                  # O(n) - works on any list
    for i, v in enumerate(a):
        if v == x: return i
    return -1

def binary_search(a, x):                  # O(log n) - REQUIRES SORTED data
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] == x: return mid
        if a[mid] < x: lo = mid + 1
        else: hi = mid - 1
    return -1

def binary_search_rec(a, x, lo=0, hi=None):
    hi = len(a) - 1 if hi is None else hi
    if lo > hi: return -1
    mid = (lo + hi) // 2
    if a[mid] == x: return mid
    return binary_search_rec(a, x, mid + 1, hi) if a[mid] < x else binary_search_rec(a, x, lo, mid - 1)

data = sorted([42, 7, 19, 3, 88, 23, 56, 71])
print(data, binary_search(data, 56), binary_search_rec(data, 3), binary_search(data, 5), linear_search(data, 88))

import bisect
print(bisect.bisect_left(data, 56))       # stdlib binary search

[3, 7, 19, 23, 42, 56, 71, 88] 5 0 -1 7
5


## 6. Sorting algorithms

| Algorithm | Best | Average | Worst | Space | Stable | Idea |
|---|---|---|---|---|---|---|
| Bubble | O(n) | O(n²) | O(n²) | O(1) | ✅ | swap adjacent out-of-order pairs |
| Selection | O(n²) | O(n²) | O(n²) | O(1) | ❌ | pick the min, put in place |
| Insertion | O(n) | O(n²) | O(n²) | O(1) | ✅ | insert each item into the sorted prefix |
| Merge | O(n log n) | O(n log n) | O(n log n) | O(n) | ✅ | divide, sort halves, merge |
| Quick | O(n log n) | O(n log n) | O(n²) | O(log n) | ❌ | partition around a pivot |
| Heap | O(n log n) | O(n log n) | O(n log n) | O(1) | ❌ | binary heap |
| **Timsort (Python)** | **O(n)** | O(n log n) | O(n log n) | O(n) | ✅ | hybrid merge+insertion; exploits existing runs |

In [11]:
def bubble_sort(a):
    a = a[:]
    for i in range(len(a) - 1):
        swapped = False
        for j in range(len(a) - 1 - i):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]; swapped = True
        if not swapped: break
    return a

def selection_sort(a):
    a = a[:]
    for i in range(len(a)):
        m = min(range(i, len(a)), key=a.__getitem__)
        a[i], a[m] = a[m], a[i]
    return a

def insertion_sort(a):
    a = a[:]
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]; j -= 1
        a[j + 1] = key
    return a

def merge_sort(a):
    if len(a) <= 1: return a
    mid = len(a) // 2
    left, right = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: out.append(left[i]); i += 1
        else: out.append(right[j]); j += 1
    return out + left[i:] + right[j:]

def quick_sort(a):
    if len(a) <= 1: return a
    pivot = a[len(a) // 2]
    return (quick_sort([x for x in a if x < pivot]) +
            [x for x in a if x == pivot] +
            quick_sort([x for x in a if x > pivot]))

import random
random.seed(3)
sample = random.sample(range(100), 12)
print(sample)
for f in (bubble_sort, selection_sort, insertion_sort, merge_sort, quick_sort):
    assert f(sample) == sorted(sample), f.__name__
    print(f"{f.__name__:15}", f(sample))

[30, 75, 69, 16, 47, 77, 60, 80, 74, 8, 1, 33]
bubble_sort     [1, 8, 16, 30, 33, 47, 60, 69, 74, 75, 77, 80]
selection_sort  [1, 8, 16, 30, 33, 47, 60, 69, 74, 75, 77, 80]
insertion_sort  [1, 8, 16, 30, 33, 47, 60, 69, 74, 75, 77, 80]
merge_sort      [1, 8, 16, 30, 33, 47, 60, 69, 74, 75, 77, 80]
quick_sort      [1, 8, 16, 30, 33, 47, 60, 69, 74, 75, 77, 80]


### Racing them

In [12]:
import time
random.seed(1)
arr = [random.randint(0, 10**6) for _ in range(2000)]
for f in (bubble_sort, selection_sort, insertion_sort, merge_sort, quick_sort, sorted):
    t = time.perf_counter(); f(arr); dt = time.perf_counter() - t
    print(f"{f.__name__:15} {dt*1000:9.2f} ms")

bubble_sort        305.37 ms
selection_sort     135.95 ms
insertion_sort     147.78 ms
merge_sort           8.56 ms
quick_sort           7.17 ms
sorted               0.42 ms


## 7. Problem-solving patterns

In [13]:
# Two pointers: pair with target sum in a SORTED list
def pair_sum(a, target):
    i, j = 0, len(a) - 1
    while i < j:
        s = a[i] + a[j]
        if s == target: return a[i], a[j]
        i, j = (i + 1, j) if s < target else (i, j - 1)
print(pair_sum([1, 3, 4, 6, 8, 11], 14))

# Sliding window: max sum of k consecutive items, O(n)
def max_window(a, k):
    cur = best = sum(a[:k])
    for i in range(k, len(a)):
        cur += a[i] - a[i - k]
        best = max(best, cur)
    return best
print(max_window([2, 1, 5, 1, 3, 2, 9, 1], 3))

# Prefix sums: range-sum queries in O(1)
from itertools import accumulate
pre = [0] + list(accumulate([3, 1, 4, 1, 5, 9]))
print(pre[5] - pre[2])       # sum of a[2:5] = 4+1+5

# Hash map for O(n) lookup: first non-repeating character
from collections import Counter
def first_unique(s):
    c = Counter(s)
    return next((ch for ch in s if c[ch] == 1), None)
print(first_unique("swiss"))

# Kadane's algorithm: maximum subarray sum
def kadane(a):
    best = cur = a[0]
    for x in a[1:]:
        cur = max(x, cur + x); best = max(best, cur)
    return best
print(kadane([-2, 1, -3, 4, -1, 2, 1, -5, 4]))

# Balanced brackets with a stack
def balanced(s):
    pairs, stack = {")": "(", "]": "[", "}": "{"}, []
    for ch in s:
        if ch in "([{": stack.append(ch)
        elif ch in pairs:
            if not stack or stack.pop() != pairs[ch]: return False
    return not stack
print(balanced("{[()]}"), balanced("{[(])}"))

# Sieve of Eratosthenes: all primes below n in O(n log log n)
def sieve(n):
    is_p = [True] * (n + 1); is_p[0:2] = [False, False]
    for i in range(2, int(n ** 0.5) + 1):
        if is_p[i]:
            is_p[i*i::i] = [False] * len(is_p[i*i::i])
    return [i for i, p in enumerate(is_p) if p]
print(sieve(50))

(3, 11)
14
10
w
6
True False
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]


---
# 🕵️ Hidden Facts & Pro Tricks

In [14]:
# 1. Raise the recursion limit (carefully) or convert recursion to a loop/explicit stack
import sys
def depth(n): return 0 if n == 0 else 1 + depth(n - 1)
old = sys.getrecursionlimit()
sys.setrecursionlimit(5000)
print(depth(3000))
sys.setrecursionlimit(old)

# 2. Iterative DFS with an explicit stack avoids RecursionError
def dfs(graph, start):
    seen, stack, order = set(), [start], []
    while stack:
        node = stack.pop()
        if node in seen: continue
        seen.add(node); order.append(node)
        stack.extend(reversed(graph[node]))
    return order
g = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": []}
print(dfs(g, "A"))

# 3. functools.cache turns exponential into linear with one decorator (see fib above)
# 4. heapq.nlargest(k, data) is better than sorting everything when k is small
import heapq
print(heapq.nlargest(3, [5, 1, 9, 3, 7]))

# 5. sorted() is stable: sort by secondary key first, then by primary key
people = [("B", 25), ("A", 25), ("C", 20)]
people.sort(key=lambda p: p[0]); people.sort(key=lambda p: p[1])
print(people)

# 6. Use `bisect` to keep a list sorted while inserting: O(log n) search
# 7. Counting sort/bucket ideas: when values are small ints, use a list of counts
def counting_sort(a, k):
    counts = [0] * (k + 1)
    for x in a: counts[x] += 1
    return [i for i, c in enumerate(counts) for _ in range(c)]
print(counting_sort([3, 1, 4, 1, 5, 2, 3], 5))

# 8. Memoize with a dict inside a closure for unhashable args (convert lists to tuples)
# 9. `math.comb`, `math.perm`, `itertools.combinations` beat hand-rolled recursion for combinatorics
# 10. Tail-call elimination is NOT done by Python — Guido rejected it to keep tracebacks readable

# 11. Amortised O(1): list.append occasionally reallocates but averages O(1)
# 12. Space-time trade-off: extra memory (a dict/set) often turns O(n^2) into O(n)
def has_duplicates_slow(a): return any(a[i] == a[j] for i in range(len(a)) for j in range(i))
def has_duplicates_fast(a): return len(set(a)) != len(a)
print(has_duplicates_slow([1, 2, 3, 1]), has_duplicates_fast([1, 2, 3]))

3000
['A', 'B', 'D', 'C']
[9, 7, 5]
[('C', 20), ('A', 25), ('B', 25)]
[1, 1, 2, 3, 3, 4, 5]
True False


**Hidden fact:** Python's sort algorithm, **Timsort** (Tim Peters, 2002), is used in Java (objects), Android and V8 too. It looks for existing sorted "runs" — sorting an already-sorted list of a million items takes milliseconds.

**Hidden fact:** the recursion limit protects the C stack. Each Python-level call in CPython 3.11+ uses less C stack, but the default of 1000 is still enforced for pure Python recursion.

**Hidden fact:** the number of moves for Tower of Hanoi with 64 disks is 2⁶⁴−1 ≈ 1.8×10¹⁹ — at one move per second the legend's temple would outlive the Sun.

---
# 📝 Exam Corner

**Theory**
1. Define recursion; base case; advantages/disadvantages; recursion vs iteration.
2. What causes `RecursionError`?
3. Explain binary search (precondition, complexity). Trace it on a list.
4. Explain bubble / selection / insertion / merge / quick sort with time complexity.
5. What is Big-O? Explain O(1), O(n), O(log n), O(n²).
6. What is memoisation? Dynamic programming?

**Trace by hand (typical question):** sort `[5, 2, 9, 1]` with bubble sort, pass by pass:
- Pass 1: 2 5 1 9  → Pass 2: 2 1 5 9 → Pass 3: 1 2 5 9.

**Output prediction**

In [15]:
def f(n):
    if n == 0:
        return 0
    return n + f(n - 1)
print(f(5))

def g(n):
    if n > 0:
        g(n - 1)
        print(n, end=" ")
g(4)
print()

def h(n):
    if n > 0:
        print(n, end=" ")
        h(n - 1)
h(4)
print()

def mystery(a, b):
    return a if b == 0 else mystery(b, a % b)
print(mystery(48, 18))

15
1 2 3 4 
4 3 2 1 
6


<details><summary>💡 Answers</summary>

```
15
1 2 3 4
4 3 2 1
6
```
Printing *after* the recursive call (`g`) reverses the order compared with printing *before* it (`h`). `mystery` is Euclid's GCD.
</details>

---
## ✅ Summary
- Recursion = base case + smaller sub-problem; the call stack holds pending work.
- Memoisation / DP removes repeated work (`@cache`).
- Big-O describes growth; know the table; prefer sets/dicts for lookup.
- Binary search needs sorted data (O(log n)); Timsort (`sorted`) is what you should use in practice.

➡️ **Next:** Module 16 — Data Structures from Scratch.